In [1]:
import os
os.environ["JAVA_HOME"] = "/Library/Java/JavaVirtualMachines/temurin-23.jdk/Contents/Home"  # or wherever your JDK 17/21 lives
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [2]:
from pyspark.sql import SparkSession
spark = (
    SparkSession.builder.appName("Local Spark Session")
    .master("local[*]")
    .config("spark.sql.warehouse.dir", "spark-warehouse")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .enableHiveSupport()
    .getOrCreate()
)
print(spark.sparkContext.uiWebUrl)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/19 13:42:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://127.0.0.1:4040


# How Spark runs on a cluster

Most of the clusters, or single machines have a Cluster manager, or some program that replicates it logic. It's a software that is responsible to share the resource of a physical machine to the processes that run on those machines.
For example, when we run a browser in a local machine, some software requested RAM memory and CPU power in order to run the Browser. These are not correlated to Spark Driver and Workers but a standalone process that every computer contains.

When we talk about Spark, Spark supports 3 types of clusters manager:
* a simple built-in standalone cluster manager; This is the one we use when we install Spark on our local machine for example;
* Apache Mesos;
* Hadoop YARN;

## Spark execute modes
We have three possible modes to run Spark:
* Cluster mode;
* Client mode;
* Local mode;

### Cluster mode
In Spark cluster mode, you submit an application (JAR, Python script, or R script) to a cluster manager. It launches and manages the driver and executor processes on worker nodes in the cluster.
![alt text](image.png)

### Client mode
In Spark client mode, the main difference is that the driver runs on the machine that submitted the application, often a gateway or edge node outside of the cluster. The cluster manager runs and manages the executors on cluster workers, while the client machine maintains the driver.
![alt text](image-1.png)

### Local mode
In spark local mode, the entire application runs on one machine, using threads for parallelism. It's useful for learning, testing and development, but not recommended for production.


## The Life Cycle of a Spark Application (Outside Spark)
A spark application typically goes through four stages in cluster mode:
* **Submit:** spark-submit - asks the cluster manager for resources to run the driver. Once the driver is placed on a cluster node, the submitting client can exit;
* **Launch:** The driver initializes a `SparkSession` and requests executors from the cluster manager, using the configurations provided at submissions;
* **Execute:** The driver schedules tasks across the executors, which process data and report task status back to the driver;
* **Complete:** The driver exist with success or failure, and the cluster manager shuts down its executors. The application's result can then be checked with the cluster manager.

## The Life Cycle of a Spark Application (Inside Spark)
A spark application begins by creating a `SparkSession`, typically with its builder and `getOrCreate()`. The session provides access to the Spark's APIs and configuration, including the lower-level `SparkContext` for RDDs, accumulators, and broadcast variables. Prefer `SparkSession` over manually creating contexts; `SQL Context` and `Hive Context` are legacy APIs, and Spark jobs generally run one at a time, unless you explicitly use threads to run actions in parallel.


## How Spark turns code into execution
Spark transformations - such as `select`, `repartition`, and `joins` are generally lazy: they describe work rather than immediately running it. For DataFrames, Spark builds a logical plan, optimizes it, and chooses a physical plan. An **action** triggers execution. For example, the `collect()` runs the plan and returns the final result to the driver.

One example creates two Spark Dataframes with two ranges, repartitions them, multiplies one ser of IDs by five, joins the sets, and sums the matching IDs. Spark's exact plan and stage/task counts can vary by version, configuration, and runtime optimizations.

### Jobs, stages, and tasks
* Job: Work triggered by an action. One action often corresponds to one job, but actions can trigger multiple jobs in some cases;
* Stage: A group of tasks that can run without waiting for an intervening shuffle. Shuffles - such as those needed for repartitiong or joining - move data between executors and typically create stage boundaries.
* Task: The unit of work Spark runs on a partition, usually on an executor. A stage's task count generally reflects the partitions it processes.

In the spark 2.2 example, each `range` starts with eight partitions, `repartition(5)` and `repartition(6)` introduce shuffles, and the join uses 200 shuffle partitions by default. The final aggregation, combines partial sums into a single result. These counts are illustrative, not guaranteed: partition settings and features such as Adaptive Query Execution can change them.

More partitions can enable more parallel work, but they also add scheduling and data-movement overhead. Too few partitions can limit parallelism; too many can make tasks inefficient. Also, `collect()` brings result to the driver, so it is best reserved for results small enough to fit in the driver memory.


### Pipelining
Spark collapses consecutive operations that don't need to move data across nodes (map -> filter, or select -> filter -> select in DataFrames) into a single stage. Each task reads a record and pushes it through all the steps without writing intermediate results anywhere. This is much faster than materializing after every step, as MapReduce did.
It's automatic. You only see it in the Spark UI or logs, as several operations folded into one stage.



### Shuffle persistence
Operations that need data from many nodes (e.g. reduce-by-key) break pipelining and force a shuffle.
Source tasks write shuffle files to their local disks, and the next stage’s tasks fetch the pieces they need.
This has two benefits. The downstream stage can run later than the upstream one, and a failed reduce task can be retried without rerunning all the map tasks.
A side effect is that a new job over already-shuffled data skips the pre-shuffle stages. They show as “skipped” in the UI. cache()/persist() gives you explicit control on top of this.

### Corrections and precisions
“Stable storage” is misleading. Shuffle files live on the executors’ local disks, which aren’t durable. If an executor or node dies, its shuffle files are lost unless an external shuffle service, or remote or push-based shuffle storage, is in use. Spark then recomputes the lost map outputs, which shows up as FetchFailedException and stage retries. The “retry without rerunning everything” claim holds only when the files survive.

“Skipped” stages are not a general cache. Reuse only happens within the same application, and only when the new job reuses the same shuffle dependency (the same RDD or DataFrame lineage). If you rebuild an equivalent DataFrame from scratch, the shuffle can be recomputed. The files are also cleaned up once the lineage is garbage collected or the application ends.


“In-memory computation tool” oversimplifies. The speedup over MapReduce comes mostly from pipelining and DAG-based planning, not from keeping everything in RAM. Shuffles always hit disk, and the shuffle write/read is often the most expensive part of a job.
Pipelining isn’t only an RDD feature. For DataFrames and SQL, Spark goes further with whole-stage code generation, fusing the pipelined operators into a single generated function. On Databricks, Photon does the equivalent natively. The “at and below the RDD level” wording is the book’s way of saying it applies to the whole stack.

Useful additions
Stage boundaries = shuffles. 

Narrow transformations (select, filter, map) pipeline within a stage. Wide ones (groupBy, join, repartition, distinct, orderBy) start a new stage.


Tasks per stage = partitions. A stage after a shuffle has spark.sql.shuffle.partitions tasks (default 200) unless AQE coalesces them. Too many or too few is a common tuning issue.
AQE changes the picture. With adaptive query execution (on by default in recent Spark and Databricks), Spark materializes shuffle stages, then uses the real statistics to coalesce partitions, switch to a broadcast join, or split skewed partitions. The final plan can differ from explain() before execution.


Exchange reuse. Within a single query, Spark can reuse identical shuffle or broadcast exchanges (ReusedExchange in the plan).